In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("marketing_team").getOrCreate()
from pyspark.sql.functions import sum

# setting up parameters

dbutils.widgets.text("bronze_customer_path","")
dbutils.widgets.text("bronze_order_path","")

dbutils.widgets.text("silver_customer_path","")
dbutils.widgets.text("silver_order_path","")

dbutils.widgets.text("gold_output_path","")

bronze_customer_path = dbutils.widgets.get("bronze_customer_path")
bronze_order_path = dbutils.widgets.get("bronze_order_path")

silver_customer_path = dbutils.widgets.get("silver_customer_path")
silver_order_path = dbutils.widgets.get("silver_order_path")

gold_output_path = dbutils.widgets.get("gold_output_path")

assert bronze_customer_path, "bronze_customer_path must not be empty"
assert bronze_order_path,    "bronze_order_path must not be empty"
assert silver_customer_path, "silver_customer_path must not be empty"
assert silver_order_path,    "silver_order_path must not be empty"
assert gold_output_path,     "gold_output_path must not be empty"


# call marketing_team note book

print("running marketing_team")
try:
    result=dbutils.notebook.run(
        "Marketing_Team",
        300,
        {
            "customer_path": bronze_customer_path,
            "order_path": bronze_order_path,
            "silver_customer_path": silver_customer_path,
            "silver_order_path": silver_order_path
        }
    )
    print(f"Notebook completed with result: {result}")
except Exception as e:
    raise RuntimeError(f"Marketing_Team note book failed: {e}")

#read delta files from silver
df_customer=spark.read.format("delta").load(silver_customer_path)
df_order=spark.read.format("delta").load(silver_order_path)

#join tables
df_gold=df_customer.alias("c").join(df_order.alias("o"),df_customer.customer_id==df_order.customer_id,"inner")

#count order amount for each customer
df_gold=df_gold.groupBy("c.customer_id","c.name").agg(sum("o.amount").alias("total_amount")).orderBy("c.customer_id")

#write into gold layer
df_gold.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(gold_output_path)
print(f"finished writing to gold:{gold_output_path}")
